In [1]:
import os
import sys
import json
from datetime import datetime
from langchain import hub
from langchain_chroma import Chroma
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.llms import Ollama
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain.prompts import ChatPromptTemplate

def clear_screen():
    os.system('cls' if os.name == 'nt' else 'clear')

def get_valid_pdf_path():
    while True:
        pdf_path = input("Enter the path to your PDF file: ")
        if os.path.isfile(pdf_path) and pdf_path.lower().endswith('.pdf'):
            return pdf_path
        else:
            print("Error: The specified file does not exist or is not a PDF. Please try again.")

def setup_rag_chain(pdf_path):
    try:
        loader = PyPDFLoader(pdf_path)
        pages = loader.load()
    except Exception as e:
        print(f"Error loading the PDF file: {e}")
        sys.exit(1)

    text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
    splits = text_splitter.split_documents(pages)

    vectorstore = Chroma.from_documents(documents=splits, embedding=HuggingFaceEmbeddings())
    retriever = vectorstore.as_retriever()

    system_prompt = input("Enter the system prompt (press Enter for default): ") or """You are an AI assistant tasked with answering questions based on the provided context. 
    Always strive to give accurate and helpful responses."""

    question_prompt_template = input("Enter the question prompt template (press Enter for default): ") or """Given the following context and question, provide a detailed answer:

    Context: {context}

    Question: {question}

    Answer:"""

    prompt = ChatPromptTemplate.from_messages([
        ("system", system_prompt),
        ("human", question_prompt_template),
    ])

    llm = Ollama(model='llama2')

    def format_docs(docs):
        return "\n\n".join(doc.page_content for doc in docs)

    rag_chain = (
        {"context": retriever | format_docs, "question": RunnablePassthrough()}
        | prompt
        | llm
        | StrOutputParser()
    )

    return rag_chain, retriever, format_docs

def save_qa_session(qa_pairs, pdf_path):
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    filename = f"qa_session_{timestamp}.json"
    data = {
        "pdf_file": pdf_path,
        "qa_pairs": qa_pairs
    }
    with open(filename, 'w') as f:
        json.dump(data, f, indent=2)
    print(f"Q&A session saved to {filename}")

def main():
    pdf_path = get_valid_pdf_path()
    rag_chain, retriever, format_docs = setup_rag_chain(pdf_path)
    qa_pairs = []

    while True:
        clear_screen()
        user_question = input("\nEnter your question about the PDF content (or type 'quit' to exit): ")
        
        if user_question.lower() == 'quit':
            save_qa_session(qa_pairs, pdf_path)
            print("Thank you for using the PDF Q&A system. Goodbye!")
            break

        try:
            context = format_docs(retriever.get_relevant_documents(user_question))
            print("\nRetrieved context:")
            print(context[:500] + "..." if len(context) > 500 else context)
            
            print("\nUser question:")
            print(user_question)
            
            result = rag_chain.invoke(user_question)
            print("\nAnswer:")
            print(result)

            qa_pairs.append({"question": user_question, "answer": result})
            
            input("\nPress Enter to continue...")
        except Exception as e:
            print(f"An error occurred while processing your question: {e}")
            print("Please ensure that Ollama is installed and running.")
            print("You can start Ollama by running the 'ollama' command in a terminal.")
            print("If the problem persists, check your firewall settings or try specifying the Ollama URL explicitly in the code.")
            input("\nPress Enter to continue...")

if __name__ == "__main__":
    main()

Enter the path to your PDF file:  


Error: The specified file does not exist or is not a PDF. Please try again.


Enter the path to your PDF file:  


Error: The specified file does not exist or is not a PDF. Please try again.


Enter the path to your PDF file:  index.pdf


...


Enter your question about the PDF content (or type 'quit' to exit):  total amt


Number of requested results 4 is greater than number of elements in index 1, updating n_results = 1
Number of requested results 4 is greater than number of elements in index 1, updating n_results = 1



Retrieved context:
YesLogic Pty. Ltd.
7 / 39 Bouverie St
Carlton VIC 3053
Australia
www.yeslogic.com
ABN 32 101 193 560InInvvoicoicee
Customer Name
Street
Postcode City
Country
Invoice date: Nov 26, 2016
Invoice number: 161126
Payment due: 30 days after invoice date
Description From Until Amount
Prince Upgrades & Support Nov 26, 2016 Nov 26, 2017 USD $950.00
Total USD $950.00
Please transfer amount to:
Bank account name: Yes Logic Pty Ltd
Name of Bank: Commonwealth Bank of Australia (CBA)
Bank State Branch (BSB): ...

User question:
total amt

Answer:
The total amount mentioned in the invoice is $950.00.



Enter your question about the PDF content (or type 'quit' to exit):  quit


Thank you for using the PDF Q&A system. Goodbye!
